# Agente RAG Experto en Motores de Combustión

## Proyecto Final - IA Generativa

Este notebook contiene ejemplos funcionales del agente RAG que responde preguntas sobre motores de 2 y 4 tiempos basándose en documentos técnicos indexados.

## 1. Configuración Inicial

Importamos las funciones principales del agente RAG.

In [ ]:
from agente_rag_langgraph_completo import chat_rag, vectorstore, llm, embeddings
import os
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

print("✅ Agente RAG cargado correctamente")
print(f"📚 Base de datos vectorial: ChromaDB")
print(f"🤖 LLM: Gemini (con failover automático)")
print(f"🔤 Embeddings: HuggingFace")

## 2. Ejemplo 1: Definición Básica

Pregunta simple sobre qué es un motor de 4 tiempos.

In [ ]:
# Crear sesión para esta conversación
thread_id = "ejemplo-1"

pregunta = "¿Qué es un motor de 4 tiempos?"
print(f"❓ Pregunta: {pregunta}")
print("\n" + "="*80 + "\n")

respuesta = chat_rag(pregunta, thread_id=thread_id)
print(f"✅ Respuesta:\n\n{respuesta}")

## 3. Ejemplo 2: Pregunta Comparativa

Comparar motores de 2 y 4 tiempos.

In [ ]:
# Nueva sesión
thread_id = "ejemplo-2"

pregunta = "¿Cuál es la diferencia entre motores de 2 y 4 tiempos?"
print(f"❓ Pregunta: {pregunta}")
print("\n" + "="*80 + "\n")

respuesta = chat_rag(pregunta, thread_id=thread_id)
print(f"✅ Respuesta:\n\n{respuesta}")

## 4. Ejemplo 3: Proceso Técnico Detallado

Explicación de las fases del ciclo de 4 tiempos.

In [ ]:
# Nueva sesión
thread_id = "ejemplo-3"

pregunta = "¿Cuáles son las fases del ciclo de 4 tiempos?"
print(f"❓ Pregunta: {pregunta}")
print("\n" + "="*80 + "\n")

respuesta = chat_rag(pregunta, thread_id=thread_id)
print(f"✅ Respuesta:\n\n{respuesta}")

## 5. Ejemplo 4: Pregunta Técnica Específica

Detalles sobre la preparación del motor.

In [ ]:
# Nueva sesión
thread_id = "ejemplo-4"

pregunta = "¿Qué es la preparación del motor?"
print(f"❓ Pregunta: {pregunta}")
print("\n" + "="*80 + "\n")

respuesta = chat_rag(pregunta, thread_id=thread_id)
print(f"✅ Respuesta:\n\n{respuesta}")

## 6. Ejemplo 5: Seguimiento con Memoria Conversacional

Demostración de que el agente recuerda preguntas anteriores en la misma sesión.

In [ ]:
# Usar MISMO thread_id para demostrar memoria
thread_id = "ejemplo-5-con-memoria"

# Primera pregunta
pregunta_1 = "¿Cuáles son las fases del motor de 2 tiempos?"
print(f"❓ Pregunta 1: {pregunta_1}")
print("\n" + "-"*80 + "\n")

respuesta_1 = chat_rag(pregunta_1, thread_id=thread_id)
print(f"✅ Respuesta 1:\n\n{respuesta_1}")
print("\n" + "="*80 + "\n")

# Segunda pregunta (con memoria de la primera)
pregunta_2 = "¿Y cuál es la diferencia con los 4 tiempos?"
print(f"\n❓ Pregunta 2 (recuerda pregunta 1): {pregunta_2}")
print("\n" + "-"*80 + "\n")

respuesta_2 = chat_rag(pregunta_2, thread_id=thread_id)
print(f"✅ Respuesta 2:\n\n{respuesta_2}")
print("\n" + "="*80 + "\n")
print("\n💡 Nota: El agente usó el mismo thread_id y recordó la pregunta anterior.")

## 7. Cómo Funciona el Agente RAG

### Componentes:

1. **Embeddings (HuggingFace)**: Convierte texto en vectores numéricos
2. **ChromaDB**: Base de datos vectorial con 632 párrafos indexados
3. **Retrieval**: Busca los 3 párrafos más relevantes para cada pregunta
4. **Generación**: Gemini combina contexto + pregunta para responder
5. **Memoria**: LangGraph guarda historial por `thread_id`

### Flujo de una pregunta:

```
Usuario pregunta:
    ↓
Convertir a vector (HuggingFace)
    ↓
Buscar en ChromaDB (k=3 párrafos más similares)
    ↓
Recuperar contexto
    ↓
Enviar a Gemini: [system_prompt + contexto + pregunta]
    ↓
Gemini genera respuesta
    ↓
Guardar en memoria (thread_id)
    ↓
Devolver respuesta al usuario
```

## 8. Sistema de Failover Automático

El agente implementa un sistema de failover que intenta múltiples modelos Gemini automáticamente si uno falla.

### ¿Qué es Failover?

Failover = cambiar automáticamente a un plan B cuando falla el plan A.

### Orden de intentos:

1. **gemini-2.5-flash** (modelo principal - rápido y barato)
2. **gemini-2.0-flash** (alternativa rápida)
3. **gemini-1.5-pro** (último recurso - más potente)

### ¿Cuándo se activa?

Si el modelo actual devuelve un error de cuota (código 429 RESOURCE_EXHAUSTED):

```
Usuario hace pregunta
    ↓
Intenta con gemini-2.5-flash
    ↓
❌ Error: Cuota excedida
    ↓
Intenta automáticamente con gemini-2.0-flash
    ↓
✅ Éxito: Devuelve respuesta
```

### Ventaja

- **Sin intervención:** El usuario no ve que cambió de modelo
- **Robusto:** Si un modelo falla por cuota, intenta el siguiente
- **Transparente:** Responde igual que si funcionara el primero

In [ ]:
# Ejemplo: El failover en acción
print("Comportamiento del failover:")
print()
print("Escenario 1: gemini-2.5-flash tiene cuota disponible")
print("  → Usuario hace pregunta")
print("  → Intenta gemini-2.5-flash ✅")
print("  → Devuelve respuesta")
print()
print("Escenario 2: gemini-2.5-flash agota cuota")
print("  → Usuario hace pregunta")
print("  → Intenta gemini-2.5-flash ❌ (Cuota excedida)")
print("  → Intenta gemini-2.0-flash ✅")
print("  → Devuelve respuesta (usuario no notó el cambio)")
print()
print("Escenario 3: Todos los modelos agotan cuota")
print("  → Usuario hace pregunta")
print("  → Intenta gemini-2.5-flash ❌")
print("  → Intenta gemini-2.0-flash ❌")
print("  → Intenta gemini-1.5-pro ❌")
print("  → Devuelve: 'Sorry! No es posible responder...'")
print()
print("💡 Cada modelo tiene su propia cuota diaria (limitada en plan free)")

## 9. Parámetros Personalizables

Puedes ajustar el comportamiento del agente modificando estos parámetros en `agente_rag_langgraph_completo.py`:

In [ ]:
# Parámetros ajustables:

parametros = """
1. k (número de documentos recuperados):
   - Actual: 3
   - Aumentar para más contexto (pero menos precisión)
   - Disminuir para respuestas más directas

2. temperature (creatividad del LLM):
   - Actual: 0.0 (factual, determinista)
   - Rango: 0.0 a 1.0
   - Para más creatividad: 0.5 o superior

3. Modelos de failover (en orden):
   - gemini-2.5-flash (rápido, barato)
   - gemini-2.0-flash (alternativa rápida)
   - gemini-1.5-pro (más potente, más lento)

4. Chunk size (tamaño de párrafos):
   - Actual: 500 caracteres
   - Mayor = contexto más amplio
   - Menor = contexto más específico
"""

print(parametros)

## 10. Verificación de la Base de Datos

Información sobre la base de datos vectorial.

In [ ]:
if vectorstore:
    print("✅ ChromaDB cargado correctamente")
    print(f"📍 Ruta: ./chroma_db_motores/")
    print(f"📄 Documentos indexados: ~632 párrafos")
    print(f"📚 PDFs procesados:")
    print("   - Funcionamiento y Preparación del Motor 2T (49 pág)")
    print("   - Motores (7 pág)")
    print("   - Curso Motor 2T (8 pág)")
    print("   - Motores de Combustión Interna (94 pág)")
    print(f"\n📊 Total: 158 páginas")
    print(f"🔤 Embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
else:
    print("❌ ChromaDB no disponible")

## 11. Creando tu Propia Sesión

Puedes crear nuevas sesiones con identifiers únicos para mantener conversaciones separadas.

In [ ]:
# Crear una sesión propia
mi_sesion = "mi-sesion-personal"

# Primera pregunta
print("Conversación 1:")
r1 = chat_rag("¿Cómo funciona la admisión en un motor?", thread_id=mi_sesion)
print(f"Respuesta: {r1[:200]}...")  # Primeros 200 caracteres

print("\n" + "-"*80 + "\n")

# Segunda pregunta (con memoria)
print("Conversación 2 (con memoria de Conversación 1):")
r2 = chat_rag("¿Qué presión hay durante este proceso?", thread_id=mi_sesion)
print(f"Respuesta: {r2[:200]}...")

## 12. Manejo de Errores

El agente maneja automáticamente ciertos errores.

In [ ]:
# El agente NO inventa si no encuentra información
pregunta_fuera_dominio = "¿Cuál es la capital de Francia?"
respuesta = chat_rag(pregunta_fuera_dominio, thread_id="error-test")
print(f"Pregunta fuera del dominio: {pregunta_fuera_dominio}")
print(f"Respuesta: {respuesta}")

print("\n" + "-"*80 + "\n")

# Si la cuota de todos los modelos se agota
print("Si todos los modelos agotan su cuota:")
print("→ El agente devuelve un mensaje de error claro (no crash)")
print("→ Se intenta automáticamente: gemini-2.5 → 2.0 → 1.5-pro")

## 13. Conclusiones

Este notebook demuestra que el agente RAG:

✅ **Responde con precisión** sobre motores de combustión  
✅ **Mantiene memoria** de conversaciones anteriores  
✅ **Fundamenta respuestas** en documentos reales  
✅ **Maneja errores** gracefully (sin crashes)  
✅ **Escala automáticamente** entre modelos Gemini (failover)  
✅ **Es reproducible** (mismo thread_id = mismo contexto)  

Para más información, ver: **README.md**